# Term Project — Task 3: Image Retrieval (Embedding Similarity Search)

학번 202502204

**추론 전용 노트북.** test 이미지를 CLIP 임베딩으로 변환하고, 각 이미지에 대해 코사인 유사도 기준 Top-K 최근접 이미지를 검색한다. 검색된 이미지의 style/fruit 라벨은 Task1 분류기(task1.pt)로 예측한다.

- 갤러리 = test 집합 자신(자기 자신 제외 Top-K).
- 출력 `./release/202502204.test.task3.txt`: query id 오름차순, 줄마다 `[(id,style_label,fruit_label), ...]`
- `TOP_K` 는 평가 시 임의 변경 가능.

In [ ]:
!pip install -qqq gdown==5.2.0
!pip install -qqq transformers==4.48.3

In [ ]:
# ===== 설정 =====
STUDENT_ID = "202502204"
TOP_K = 5                       # 평가 시 임의 변경 (ex. 3, 5, 10)

# 라벨링에 재사용할 Task1 모델(task1.pt)의 Google Drive 파일 ID
MODEL_FILE_ID = "12bq6yp7NkHreXwQZz6zH73bNrwu5zZSG"

DATA_ROOT = "data/test"
TEST_ZIP_ID = ""                # 비우면 data/test/images 가 이미 존재해야 함
CLIP_NAME = "openai/clip-vit-base-patch32"
BATCH_SIZE = 64

In [ ]:
import os, zipfile, gdown

def has_images(root):
    if not os.path.isdir(root):
        return False
    for _, _, fs in os.walk(root):
        if any(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in fs):
            return True
    return False

if not has_images(DATA_ROOT):
    if TEST_ZIP_ID:
        gdown.download(f"https://drive.google.com/uc?id={TEST_ZIP_ID}", "test.zip", quiet=False)
        os.makedirs("data", exist_ok=True)
        with zipfile.ZipFile("test.zip") as z:
            z.extractall("data")
    assert has_images(DATA_ROOT), f"test 이미지를 {DATA_ROOT}/images/ 에 두거나 TEST_ZIP_ID 설정"

MODEL_PATH = "task1.pt"
if not os.path.exists(MODEL_PATH):
    gdown.download(f"https://drive.google.com/uc?id={MODEL_FILE_ID}", MODEL_PATH, quiet=False)
print("data ok:", DATA_ROOT, "| model:", MODEL_PATH)

In [ ]:
# --- test 이미지 수집 (재귀, id 숫자 정렬) ---
from glob import glob
EXTS = (".jpg", ".jpeg", ".png")
files = [p for p in glob(os.path.join(DATA_ROOT, "**", "*"), recursive=True)
         if p.lower().endswith(EXTS)]
def id_key(p):
    stem = os.path.splitext(os.path.basename(p))[0]
    return (0, int(stem)) if stem.isdigit() else (1, stem)
files = sorted(set(files), key=id_key)
ids = [os.path.basename(p) for p in files]
print("test images:", len(files))

In [ ]:
# --- CLIP 임베딩 (코사인용 L2 정규화) ---
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPModel.from_pretrained(CLIP_NAME).to(device).eval()
proc = CLIPProcessor.from_pretrained(CLIP_NAME)

@torch.no_grad()
def embed(paths):
    imgs = [Image.open(p).convert("RGB") for p in paths]
    batch = proc(images=imgs, return_tensors="pt").to(device)
    feat = clip.get_image_features(**batch)
    return torch.nn.functional.normalize(feat, dim=-1)

embs = []
for i in range(0, len(files), BATCH_SIZE):
    embs.append(embed(files[i:i + BATCH_SIZE]).cpu())
embs = torch.cat(embs)  # (N, D), normalized
print("embeddings:", tuple(embs.shape))

In [ ]:
# --- Task1 분류기로 style/fruit 라벨 예측 ---
import torch.nn as nn
from torchvision import models, transforms

class DualHeadClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        b = models.efficientnet_b0(weights=None)
        inf = b.classifier[1].in_features; b.classifier = nn.Identity()
        self.backbone = b; self.dropout = nn.Dropout(0.2)
        self.fruit_head = nn.Linear(inf, 6); self.style_head = nn.Linear(inf, 3)
    def forward(self, x):
        f = self.dropout(self.backbone(x)); return self.fruit_head(f), self.style_head(f)

ckpt = torch.load(MODEL_PATH, map_location=device)
state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
IMG_SIZE = ckpt.get("img_size", 224) if isinstance(ckpt, dict) else 224
clf = DualHeadClassifier().to(device); clf.load_state_dict(state); clf.eval()
ctf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

@torch.no_grad()
def classify(paths):
    x = torch.stack([ctf(Image.open(p).convert("RGB")) for p in paths]).to(device)
    fp, sp = clf(x); return fp.argmax(1).cpu().tolist(), sp.argmax(1).cpu().tolist()

fruit_lab, style_lab = [], []
for i in range(0, len(files), BATCH_SIZE):
    fr, st = classify(files[i:i + BATCH_SIZE]); fruit_lab += fr; style_lab += st
print("labeled:", len(fruit_lab))

In [ ]:
# --- Top-K 검색 (자기 자신 제외) & 출력 ---
os.makedirs("release", exist_ok=True)
out_path = f"release/{STUDENT_ID}.test.task3.txt"
N = embs.size(0)
embs_d = embs.to(device)

with open(out_path, "w") as f:
    for i in range(0, N, BATCH_SIZE):
        q = embs_d[i:i + BATCH_SIZE]
        sims = q @ embs_d.t()                       # (b, N) cosine
        for r in range(q.size(0)):
            gi = i + r
            sims[r, gi] = -1.0                       # exclude self
            topk = torch.topk(sims[r], TOP_K).indices.cpu().tolist()
            items = [f"({ids[j]},{style_lab[j]},{fruit_lab[j]})" for j in topk]
            f.write("[" + ", ".join(items) + "]\n")
print("wrote", out_path)

In [ ]:
with open(out_path) as f:
    lines = f.read().splitlines()
print("lines:", len(lines), "| TOP_K =", TOP_K)
for ln in lines[:3]:
    print(ln)